# D4a 어텐션의 원리 — 실습 (W10, D4 2부작 1편)

> ⚠️ **가장 먼저 — 화면 위 [Drive로 복사]를 누르세요.**
> 지금 보고 있는 것은 원본을 잠깐 띄운 **임시 사본**입니다. 복사하지 않으면 탭을 닫는 순간
> 채운 빈칸과 실행 결과가 **모두 사라집니다.** 복사본은 내 Google Drive에 저장되고,
> 원본은 바뀌지 않으니 마음껏 고쳐도 됩니다.

> 위에서부터 한 셀씩 `Shift+Enter`로 실행하세요. `___` 빈칸은 직접 채웁니다.
> (20 Newsgroups 데이터가 처음 실행 시 자동 다운로드됩니다.)

**이 실습이 끝나면**
1. 손계산 결과 [0.09, 0.67, 0.24] → [0.18, 1.58]를 **torch로 검증**한다
2. 점수 크기와 softmax **쏠림**의 관계를 실험한다 (√dₖ의 이유)
3. 완성형 어텐션 함수 + 4단어 **히트맵**(royalty/fruit 블록)을 그린다
4. Q·K·V **투영**과 `nn.MultiheadAttention`을 체험한다
5. **어텐션 뉴스 분류기**로 D3b의 74% → **82.3%** + 판단 근거를 본다

**7단계 멘탈모델 초점:** 모델(정보를 어떻게 모을 것인가)

## Part A. 세상에서 가장 작은 어텐션 — 손계산 검증 ⭐
단어 3개, 2차원 [왕족성, 과일성]. Q=K=V=X, 스케일링 생략(손계산 그대로).
**먼저 종이에서 orange 행을 완주한 뒤** 실행해 답을 맞춰 보세요.

In [ ]:
import torch                                          # PyTorch
import torch.nn as nn                                 # 신경망 모듈
import matplotlib.pyplot as plt                       # 그래프

words3 = ['king', 'apple', 'orange']                  # 미니 단어장
X3 = torch.tensor([[2.0, 0.0],                        # king  = [왕족성 2, 과일성 0]
                   [0.0, 2.0],                        # apple = [0, 2]
                   [0.0, 1.0]])                       # orange= [0, 1]

scores3 = X3 @ X3.T                                   # ① 내적 점수 (Q=K=X)
print('scores:'); print(scores3)                      # orange 행 = [0, 2, 1] 확인!

attn3 = torch.softmax(scores3, dim=___)               # ✍️ 빈칸: ② 각 행(Query)별 합=1 → 마지막 차원
out3 = attn3 @ X3                                     # ③ 가중치로 Value(=X) 가중합
for i, w in enumerate(words3):                        # 행별 출력
    print(w.ljust(7), '주목', [round(v, 2) for v in attn3[i].tolist()],
          '→ 새 표현', [round(v, 4) for v in out3[i].tolist()])

> **검산 포인트:** orange 행 = [0.09, 0.67, 0.24], 새 표현 = [0.1801, 1.5752] — 손계산과 일치?
> orange는 **자기 자신(0.24)보다 apple(0.67)에 더 주목**했고, 그 결과 과일성이 1 → 1.58로 진해졌습니다(문맥 반영). 반면 king은 자기 자신에 0.96 — **대각선의 지배**(투영이 필요한 이유).

## Part B. 쏠림 실험 — √dₖ로 나누는 이유
점수의 **크기**가 softmax를 어떻게 바꾸는지 orange 행으로 확인합니다.

In [ ]:
for name, s in [('원점수     ', scores3),             # 그대로
                ('점수 ÷ √2  ', scores3 / (2 ** 0.5)),  # 스케일링(완화)
                ('점수 × 10  ', scores3 * 10)]:       # 점수 폭주 상황
    a = torch.softmax(s, dim=-1)[2]                   # orange 행만
    print(name, [round(v, 2) for v in a.tolist()])    # 쏠림 정도 비교

> 점수를 10배 키우면 [0, 1, 0] — **원핫 쏠림**(한 단어만 보고 나머지 무시). 내적은 차원이 클수록 커지므로 **√dₖ로 나눠** 완화합니다. D1b의 학습률처럼 "적당한 온도"가 학습을 살립니다.

## Part C. 완성형 어텐션 + 4단어 히트맵
`softmax(Q·Kᵀ/√dₖ)·V` — Scaled Dot-Product Attention을 함수로 완성합니다.

In [ ]:
def attention(Q, K, V):                               # Scaled Dot-Product Attention
    d_k = Q.size(-1)                                  # Key 차원
    scores = Q @ K.transpose(-2, -1) / (d_k ** ___)   # ✍️ 빈칸: √dₖ로 나누기 = 지수 얼마?
    attn = torch.softmax(scores, dim=-1)              # 행(Query)별 가중치
    return attn @ ___, attn                           # ✍️ 빈칸: 가중치로 무엇을 가중합?

words = ['king', 'queen', 'apple', 'orange']          # 4단어로 확장
X = torch.tensor([[3.0, 0.0, 0.3],                    # king   [royalty, fruitiness, sweetness]
                  [2.7, 0.0, 0.3],                    # queen
                  [0.0, 3.0, 2.7],                    # apple
                  [0.0, 3.0, 0.9]])                   # orange

out, attn = attention(X, X, X)                        # Self-Attention (Q=K=V=같은 문장)
print('attention weights (행=주목하는 단어, 열=주목받는 단어):')
for i, w in enumerate(words):                         # 각 단어의 주목 분포
    print(' ', w.ljust(7), [round(v, 2) for v in attn[i].tolist()])

In [ ]:
A = attn.detach().numpy()                             # (4,4) 가중치 행렬
fig, ax = plt.subplots(figsize=(4.2, 3.8))
im = ax.imshow(A, cmap='Blues', vmin=0, vmax=1)       # 진할수록 큰 가중치
ax.set_xticks(range(4)); ax.set_yticks(range(4))
ax.set_xticklabels(words, rotation=30); ax.set_yticklabels(words)
ax.set_xlabel('Key (attended to)'); ax.set_ylabel('Query (attending from)')
ax.set_title('Self-Attention Weights')                # 라벨은 영어(Colab 한글 폰트 깨짐 방지)
for i in range(4):                                    # 칸마다 수치 표기
    for j in range(4):
        ax.text(j, i, f'{A[i, j]:.2f}', ha='center', va='center',
                color='white' if A[i, j] > 0.5 else 'black', fontsize=9)
fig.colorbar(im, fraction=0.046, pad=0.04); fig.tight_layout(); plt.show()

> 좌상 **royalty 블록**(king·queen)·우하 **fruit 블록**(apple·orange)이 진하게 — 닮은 단어끼리 서로 주목. 어텐션 행렬은 "그림이 되는 수치"입니다.

## Part D. Q·K·V 투영 — 질문과 색인을 분리
실제 트랜스포머는 입력을 **학습되는 행렬 Wq·Wk·Wv**로 투영해 Q·K·V를 만듭니다.
(여기선 무작위 투영으로 '주목 분포가 달라진다'는 것만 관찰 — 실전에선 학습됨)

In [ ]:
torch.manual_seed(0)                                  # 재현성
d = X.size(-1)                                        # 임베딩 차원(3)
Wq, Wk, Wv = torch.randn(d, d), torch.randn(d, d), torch.randn(d, d)  # 학습될(여기선 무작위) 투영
out2, attn2 = attention(X @ Wq, X @ ___, X @ Wv)      # ✍️ 빈칸: Key는 어떤 행렬로 투영?
print('out shape:', tuple(out2.shape))                # (4, 3) — 모양은 그대로
print('행 합(=1):', [round(v, 2) for v in attn2.sum(-1).tolist()])
for i, w in enumerate(words):                         # 투영 후 주목 분포
    print(' ', w.ljust(7), [round(v, 2) for v in attn2[i].tolist()])

> 투영을 거치니 **대각선 지배가 사라지고** 주목 분포가 완전히 달라졌습니다(무작위라 의미는 없음). 실전에서는 이 W들이 **손실을 줄이도록 학습**되어 "it→animal" 같은 필요한 연결을 스스로 만듭니다.

## Part E. 멀티헤드 어텐션 (PyTorch 내장)
여러 head가 **서로 다른 관점**으로 동시에 주목한 뒤 결과를 이어 붙입니다.

In [ ]:
torch.manual_seed(0)                                  # 재현성
mha = nn.MultiheadAttention(embed_dim=4, num_heads=2, batch_first=___)  # ✍️ 빈칸: (배치,시간,특징) 규격(D3a!)
x = torch.randn(1, 4, 4)                              # (배치=1, 단어=4, 차원=4)
y, w = mha(x, x, x)                                   # self-attention (Q=K=V=x)
print('출력 shape:', tuple(y.shape))                  # 입력과 같은 모양
print('가중치 shape:', tuple(w.shape))                # (배치, 단어, 단어)
print('행 합:', [round(v, 2) for v in w[0].sum(-1).tolist()])  # 각 행 합=1

## Part F. 실전 — 어텐션 뉴스 분류기 (74%의 벽에 도전) ⭐
D3b와 **완전히 같은 데이터·전처리**(야구 vs 우주). LSTM 대신 **학습되는 질문 벡터 q 하나**가 "분류에 중요한 단어는?"을 묻습니다. 어텐션의 추가 비용은 **파라미터 단 64개**.

In [ ]:
import re                                             # 토큰화(D3b 그대로)
from sklearn.datasets import fetch_20newsgroups       # 뉴스 데이터
from collections import Counter                       # 단어 빈도

def tok(t):                                           # D3b의 토크나이저 그대로
    return re.findall(r'[a-z]+', t.lower())

cats = ['rec.sport.baseball', 'sci.space']            # 두 주제
tr = fetch_20newsgroups(subset='train', categories=cats, remove=('headers','footers','quotes'))
te = fetch_20newsgroups(subset='test',  categories=cats, remove=('headers','footers','quotes'))

counter = Counter(w for d in tr.data for w in tok(d)) # 빈도는 train으로만(M2 누수 방지)
itos = ['<pad>', '<unk>'] + [w for w, _ in counter.most_common(5000)]  # 예약 2 + 상위 5000
stoi = {w: i for i, w in enumerate(itos)}             # 단어→정수

MAX = 200                                             # 고정 길이(앞쪽 패딩)
def encode(t):                                        # 문서 → 길이 200 정수 시퀀스
    ids = [stoi.get(w, 1) for w in tok(t)][:MAX]      # 인코딩(+길면 자르기)
    return [0] * (MAX - len(ids)) + ids               # 앞쪽을 <pad>로

Xtr = torch.tensor([encode(d) for d in tr.data]); ytr = torch.tensor(tr.target)
Xte = torch.tensor([encode(d) for d in te.data]); yte = torch.tensor(te.target)
print('train:', tuple(Xtr.shape), '| test:', tuple(Xte.shape))  # (1190, 200) / (791, 200)

In [ ]:
torch.manual_seed(0)                                  # 재현성
class AttnClassifier(nn.Module):                      # 임베딩 → 어텐션 풀링 → Linear
    def __init__(self, vocab_size, embed=64, nclass=2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed, padding_idx=0)  # D3b 그대로
        self.query = nn.Parameter(torch.randn(embed)) # 학습되는 질문 벡터 — 단 64개!
        self.fc = nn.Linear(embed, nclass)            # 문서 벡터 → 클래스 점수
    def forward(self, x, return_attn=False):
        e = self.embedding(x)                         # (B, 200) → (B, 200, 64)
        scores = e @ self.query / (e.size(-1) ** 0.5) # ① 내적 점수 ÷ √dₖ → (B, 200)
        scores = scores.masked_fill(x == ___, -1e9)   # ✍️ 빈칸: <pad>의 번호를 가림(마스킹)
        attn = torch.softmax(scores, dim=-1)          # ② 단어별 주목 가중치(합=1)
        doc = (attn.unsqueeze(-1) * e).sum(dim=___)   # ✍️ 빈칸: ③ 시간(단어) 차원으로 가중합
        if return_attn:                               # 근거 확인용
            return self.fc(doc), attn
        return self.fc(doc)                           # (B, 2)

model = AttnClassifier(len(itos))                     # 생성
n_total = sum(p.numel() for p in model.parameters())  # 전체(D1c numel)
n_emb = sum(p.numel() for p in model.embedding.parameters())  # 임베딩만
print('전체:', f'{n_total:,}', '| 임베딩:', f'{n_emb:,}',
      '| 어텐션(query): 64 | fc: 130')                # 320,322 — 사실상 전부 임베딩

In [ ]:
from torch.utils.data import TensorDataset, DataLoader  # 배치 공급

train_loader = DataLoader(TensorDataset(Xtr, ytr), batch_size=32, shuffle=True)
criterion = nn.CrossEntropyLoss()                     # 분류 손실(D1b)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)  # Adam

losses = []                                           # epoch 손실 기록
for epoch in range(5):                                # 5바퀴 (D3b와 동일 조건)
    model.train(); running = 0.0
    for xb, yb in train_loader:                       # D1c 5단계 그대로
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()
        running += loss.item()
    losses.append(running / len(train_loader))
    print(f'epoch {epoch+1}: train loss = {losses[-1]:.4f}')

model.eval()                                          # 평가 스위치(D1c)
with torch.no_grad():
    acc = (model(Xte).argmax(-1) == yte).float().mean().item()
print('test accuracy:', round(acc, 3), '(D3b LSTM: 0.741)')  # ~0.823 — 벽을 넘었다!

plt.figure(figsize=(6, 3.5))                          # 손실 곡선
plt.plot(range(1, 6), losses, 'o-')                   # epoch별
plt.xlabel('epoch'); plt.ylabel('train loss')         # 축(영어)
plt.xticks(range(1, 6))                               # 정수 눈금
plt.title(f'Attention classifier (test acc {acc:.3f})')  # 제목(영어)
plt.grid(True); plt.show()

## Part G. 진단 — 어텐션은 근거를 보여 준다
LSTM의 h는 "왜 그렇게 판단했나"에 말이 없지만, **어텐션 가중치는 판단 근거의 지도**입니다.

In [ ]:
with torch.no_grad():                                 # 시험 전체의 가중치 회수
    logits, attn_te = model(Xte, return_attn=True)
pred = logits.argmax(-1)                              # 예측 클래스

def top_words(i, k=8):                                # i번 문서의 최다 주목 단어 k개
    vals, pos = attn_te[i].topk(k)                    # 가중치 상위 k
    return [(itos[Xte[i][p]], round(v.item(), 3)) for v, p in zip(vals, pos)]

found = {}                                            # 클래스별 표본 문서 1개씩
for i in range(len(yte)):
    if pred[i] == yte[i] and Xte[i].bool().sum() > 50:  # 맞힌 + 충분히 긴 문서
        name = te.target_names[yte[i]]
        if name not in found:
            found[name] = i
    if len(found) == 2:
        break
for name, i in found.items():                         # 무엇을 보고 판단했나
    print(f'[{name}] 문서 top-8 주목 단어:')
    print('  ', top_words(i))

In [ ]:
i = found['sci.space']                                # 우주 문서 하나의 주목 지도
vals, pos = attn_te[i].topk(10)                       # 상위 10 단어
labels = [itos[Xte[i][p]] for p in pos]
fig, ax = plt.subplots(figsize=(6.5, 3.5))
ax.barh(range(9, -1, -1), [v.item() for v in vals])   # 막대(위=1등)
ax.set_yticks(range(9, -1, -1)); ax.set_yticklabels(labels)
ax.set_xlabel('attention weight')                     # 축(영어)
ax.set_title('Top-10 attended words (a sci.space doc)')
fig.tight_layout(); plt.show()

In [ ]:
my_sents = ['the rocket will launch into orbit tomorrow',   # 내 문장 테스트(D3b와 같은 세트)
            'he hit a home run in the ninth inning',
            'the pitcher threw the ball to the moon']        # 야구+우주 함정 문장!
with torch.no_grad():
    for s in my_sents:
        xin = torch.tensor([encode(s)])               # 같은 파이프라인으로 인코딩
        logit, a = model(xin, return_attn=True)
        p = logit.argmax(-1).item()                   # 판정
        vals, pos = a[0].topk(3)                      # 근거 top3
        tw = [(itos[xin[0][q]], round(v.item(), 2)) for v, q in zip(vals, pos)]
        print(f'"{s}" → {te.target_names[p]} | 근거 {tw}')

> 함정 문장(투수가 달로 공을 던졌다)은 space로 — **moon에 최대 주목(0.35)**. 모델이 어느 단서에 끌렸는지 가중치가 그대로 보여 줍니다. D3b에서는 불가능했던 진단입니다.

## 🤖 AI 코파일럿 활용 (선택) — ai-native v1
막히면 AI 튜터에게 묻되, **먼저 스스로 생각**하고 답을 **실행으로 검증**하세요.

**좋은 질문 예시**
- "orange 행 [0.09, 0.67, 0.24]를 내가 손으로 유도해 볼 테니 채점해 줘."
- "pad 마스킹(−1e9)을 빼면 정확도가 어떻게 될지 가설을 세웠어 — 실험 설계를 도와줘."
- "어텐션 분류기가 단어 순서를 무시하는데도 82%가 나온 이유를 설명해 볼게."
- "top-8 주목 단어에 'its' 같은 기능어도 섞여 있어 — 왜 그럴지 같이 가설을 세워 줘."

**가드레일**
1. 먼저 손으로 생각 → 그 다음 AI
2. AI 코드는 *왜 그런지* 설명할 수 있을 때만 사용
3. AI 출력은 실행으로 검증

## 정리 & 자가 점검

**오늘 한 일 3줄**
1. 어텐션 3단계(내적→softmax→가중합)를 손계산으로 완주하고 torch로 검증했다 ([0.09, 0.67, 0.24] → [0.18, 1.58])
2. 쏠림 실험(×10 → 원핫)으로 √dₖ의 이유를 확인하고, 투영·멀티헤드를 체험했다
3. 어텐션 분류기(추가 64개)로 D3b의 74% → **82.3%** + 판단 근거(moon 0.35)까지 봤다

**스스로 점검**
- [ ] `dim=-1`이 "행(Query)별 합=1"과 왜 짝인지 안다
- [ ] pad 마스킹이 없으면 무슨 일이 생기는지 안다 (e⁰=1)
- [ ] 어텐션 분류기가 순서를 모른다는 것과, 그런데도 이 과제에선 통한 이유를 안다

**🔹심화 (선택)**
- **마스킹 제거 실험:** `masked_fill` 줄을 주석 처리하고 재학습 — 정확도 변화를 실측해 보세요(pad가 몫을 챙기면?).
- **단어 섞기 실험:** `encode` 결과를 `torch.randperm`으로 뒤섞어 평가 — 정확도가 거의 안 변하는 것을 확인(순서 무지의 증거, D4b 위치 인코딩의 동기).
- **query 벡터를 2개로:** 질문 2개의 가중합을 이어 붙이면(cat) 성능이 오를까요? — 멀티헤드의 손맛.
- 히트맵의 X에서 sweetness 값을 바꿔 보세요 — 블록 구조가 어떻게 변하나.